In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q --no-cache-dir google-search-results
import importlib, sys, pkgutil, subprocess, json

  Preparing metadata (setup.py) ... done


In [ ]:
# ✅ SerpAPI via plain HTTP (works on Py 3.12 too)
!pip -q install pandas requests

import os, requests, pandas as pd

API_KEY = os.getenv("SERPAPI_API_KEY") or "6ce1429fd1015da89e796a8bac25da62d85753020327af9dcc60b19e6b4e47e0" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."

BASE = "https://serpapi.com/search.json"

def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

print("✅ Setup complete")

✅ Setup complete


In [ ]:
import time
import pandas as pd

# Define your single query
query = "gift shop"

# Common search params for Venice (adjust as needed)
BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "it",
    "google_domain": "google.it",
    "ll": "@45.437190,12.334590,13z" # Venice center + zoom
}

all_paginated_rows = []

# SerpAPI usually returns a maximum of 20 results per page for Google Maps.
# To get more, we need to paginate using the 'start' parameter.
# Let's aim for up to 100 results (5 pages of 20 results each).
MAX_RESULTS_TO_FETCH = 300
RESULTS_PER_PAGE = 20 # Max per page for google_maps engine

print(f"Fetching results for: '{query}' with pagination...")

for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
    print(f"  Fetching from start index: {start_index}")
    params = {**BASE_PARAMS, "q": query, "start": start_index, "num": RESULTS_PER_PAGE}
    try:
        res = serpapi_search("google_maps", params)
        results = res.get("local_results", []) or []

        if not results:
            print("  No more results found for this query.")
            break

        for r in results:
            gps = r.get("gps_coordinates") or {}
            all_paginated_rows.append({
                "source_query": query,
                "position": r.get("position"),
                "title": r.get("title"),
                "rating": r.get("rating"),
                "reviews": r.get("reviews"),
                "type": r.get("type"),
                "address": r.get("address"),
                "open_state": r.get("open_state"),
                "place_id": r.get("place_id"),
                "link": r.get("link"),
                "thumbnail": r.get("thumbnail"),
                "original": r.get("original"),
                "gps": gps,
                "lat": gps.get("latitude"),
                "lng": gps.get("longitude"),
            })
        time.sleep(0.8)  # Be polite, avoid hitting rate limits

    except Exception as e:
        print(f"⚠️ Query failed for '{query}' at start index {start_index}: {e}")
        break

df_paginated_places = pd.DataFrame(all_paginated_rows)
print(f"Total results fetched: {len(df_paginated_places)}")

Fetching results for: 'gift shop' with pagination...
  Fetching from start index: 0
  Fetching from start index: 20
  Fetching from start index: 40
  Fetching from start index: 60
  Fetching from start index: 80
  Fetching from start index: 100
⚠️ Query failed for 'gift shop' at start index 100: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_maps&api_key=6ce1429fd1015da89e796a8bac25da62d85753020327af9dcc60b19e6b4e47e0&type=search&hl=en&gl=it&google_domain=google.it&ll=%4045.437190%2C12.334590%2C13z&q=gift+shop&start=100&num=20
Total results fetched: 100


In [ ]:
display(df_paginated_places.head())

,source_query,position,title,rating,reviews,type,address,open_state,place_id,link,thumbnail,original,gps,lat,lng
0,gift shop,1,zacaria's,4.9,49.0,Gift shop,"C. de la Fenice, 1920, 30124 Venezia VE, Italy",Open · Closes 6:30 PM,ChIJ0SvZ7dCxfkcRv8tr9rs6qiI,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4338458, 'longitude': 12.33374...",45.433846,12.333748
1,gift shop,2,souvenir,3.7,3.0,Gift shop,"S. Marco, 4485A, 30124 Venezia VE, Italy",None,ChIJAZzyCE2xfkcR4L6r6yirAaQ,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4353592, 'longitude': 12.3357248}",45.435359,12.335725
2,gift shop,3,Pylones Venezia,4.4,126.0,Gift shop,"Strada Nova, 3948, 30121 Venezia VE, Italy",Open · Closes 7:30 PM,ChIJpUMA4-mxfkcRp98Vt5RR8kM,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4411862, 'longitude': 12.3344592}",45.441186,12.334459
3,gift shop,4,Murano Collection snc,5.0,4.0,Gift shop,"Riva degli Schiavoni, 3738, 30122 Venezia VE, ...",Open · Closes 9 PM,ChIJe7NbsFevfkcRCFoKQjF_hmA,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.433711599999995, 'longitude': ...",45.433712,12.346709
4,gift shop,5,Souvenir da Gabriele Al numero 744,4.7,15.0,Gift shop,"Ponte Cavagnis, 744, 30135 Venezia VE, Italy",None,ChIJ3W3Ho6CxfkcR8WpQ21FnqIc,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4403278, 'longitude': 12.3235824}",45.440328,12.323582


## Combine Restaurant Details with Reviews

In [ ]:
import pandas as pd

# First, ensure df_results has a 'place_id' column for merging
# The PLACE_ID variable holds the ID for which df_results was fetched.
# If df_results is empty or the PLACE_ID is None, this step handles it gracefully.

# Initialize df_results and PLACE_ID if they are not defined
if 'df_results' not in locals() and 'df_results' not in globals():
    df_results = pd.DataFrame()
if 'PLACE_ID' not in locals() and 'PLACE_ID' not in globals():
    PLACE_ID = None

# Ensure df_results has a 'place_id' column if it's an empty DataFrame with no columns.
# This is crucial to prevent KeyError during the merge operation when df_results is empty.
if df_results.empty and not df_results.columns.isin(['place_id']).any():
    df_results = pd.DataFrame(columns=['place_id'])

# If df_results is not empty and PLACE_ID is defined, and 'place_id' column doesn't exist, add it.
# This part handles the case where review data *was* fetched for a single PLACE_ID
# but the 'place_id' column wasn't explicitly set during fetching.
if not df_results.empty and PLACE_ID and 'place_id' not in df_results.columns:
    df_results['place_id'] = PLACE_ID

# Perform a left merge to combine restaurant details with reviews
# A left merge keeps all rows from df_paginated_places and adds matching review data.
# 'suffixes' are used to distinguish columns with the same name (like 'rating' and 'title')
# coming from both DataFrames.
df_combined_data = pd.merge(
    df_paginated_places,
    df_results,
    on='place_id',
    how='left',
    suffixes=('_place', '_review')
)

# Display the entire combined DataFrame
display(df_combined_data)

,source_query,position,title,rating,reviews,type,address,open_state,place_id,link,thumbnail,original,gps,lat,lng
0,supermarket,1,Conad,4.2,876.0,Supermarket,"SESTIERE SANTA CROCE, 892, 30135 Venezia VE, I...",Closed · Opens 8:30 AM,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.439881299999996, 'longitude': ...",45.439881,12.324617
1,supermarket,2,Coop,4.1,2035.0,Supermarket,"San Marco, Riva del Carbon, 4173/4177, 30124 V...",Closed · Opens 8:30 AM,ChIJr6rjRNqxfkcR6MWYsM1pwTI,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4364907, 'longitude': 12.3343808}",45.436491,12.334381
2,supermarket,3,Supermercato Coop,4.1,2185.0,Supermarket,"Santa Croce, 499, 30121 Venezia VE, Italy",Closed · Opens 8:30 AM,ChIJz9iqFrixfkcRK5g8t1KreiE,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.439554, 'longitude': 12.317910...",45.439554,12.317910
3,supermarket,4,Conad City - Supermarket,4.4,752.0,Supermarket,"Sestiere Cannaregio, 3027, 30100 Venezia VE, I...",Closed · Opens 7 AM,ChIJl14O4eqxfkcRTSK1pzkmshk,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4480222, 'longitude': 12.3253576}",45.448022,12.325358
4,supermarket,5,Despar Teatro Italia,4.5,4048.0,Supermarket,"Cannaregio nn, Campiello de l'Anconeta, 1939-1...",Closed · Opens 8:30 AM,ChIJlYIMP8KxfkcRHcslK4IM_JU,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4440769, 'longitude': 12.3294959}",45.444077,12.329496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,supermarket,16,i&s Farm ll Biologico di Sant'Erasmo,5.0,17.0,Organic food store,"Corte dei Pali, 3818, 30100 Venezia VE, Italy",Closed · Opens 8:30 AM,ChIJFWEtfQ2xfkcR-fjNV3mrZ9o,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4415319, 'longitude': 12.3339252}",45.441532,12.333925
96,supermarket,17,Imperial Lido Fruits,4.9,8.0,Greengrocer,"Via Sandro Gallo, 135/e, 30126 Lido VE, Italy",None,ChIJgxH9S0-vfkcRKDV8XuJl5Bs,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.403896499999995, 'longitude': ...",45.403896,12.363072
97,supermarket,18,"Banchetto frutta e verdura ""di Thomas""",5.0,2.0,Supermarket,"Campiello Santa Maria Formosa, 5240, 30122 Ven...",Closed · Opens 8 AM,ChIJrYJRbwCxfkcRaffYOmjwm9c,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.437326399999996, 'longitude': ...",45.437326,12.341163
98,supermarket,19,COOP BURANO,4.3,171.0,Supermarket,"Via San Mauro, 260, 30142 Venezia VE, Italy",Closed · Opens 8:30 AM,ChIJd_DexCmtfkcROX_HkWNSa0E,None,https://lh3.googleusercontent.com/gps-cs-s/AG0...,None,"{'latitude': 45.4861363, 'longitude': 12.4151765}",45.486136,12.415176


In [ ]:
# 💾 Save your DataFrame: change df_name and path
df_name = 'df_combined_data'   # replace with the variable name you want to save
OUTPATH = "/content/drive/MyDrive/semester1_python_dataset/Google/Data_Scratch/supermarket.csv"
globals()[df_name].to_csv(OUTPATH, index=False)
OUTPATH

'/content/drive/MyDrive/semester1_python_dataset/Google/Data_Scratch/supermarket.csv'

In [ ]:
import folium

# Define variables for map centering and boundaries using df_paginated_places
df = df_paginated_places # Use the paginated DataFrame for mapping

# --- Filter out obvious outliers for Venice coordinates ---
# Approximate bounding box for Venice, Italy
VENICE_LAT_MIN, VENICE_LAT_MAX = 45.3, 45.5
VENICE_LNG_MIN, VENICE_LNG_MAX = 12.2, 12.4

df_filtered = df[
    (df['lat'] >= VENICE_LAT_MIN) & (df['lat'] <= VENICE_LAT_MAX) &
    (df['lng'] >= VENICE_LNG_MIN) & (df['lng'] <= VENICE_LNG_MAX)
].copy()

# If after filtering, the DataFrame is empty, use the original (unfiltered) or handle appropriately
if df_filtered.empty:
    print("Warning: No points found within Venice's approximate bounds. Using unfiltered data.")
    df_to_map = df
else:
    df_to_map = df_filtered

centre_latitude = df_to_map['lat'].mean()
centre_longitude = df_to_map['lng'].mean()

# --- Customize the bounding box coordinates here ---
# Current data range from filtered data:
# lat_north_auto = df_to_map['lat'].max()
# lat_south_auto = df_to_map['lat'].min()
# long_east_auto = df_to_map['lng'].max()
# long_west_auto = df_to_map['lng'].min()

# Example: slightly expanded bounds. Adjust these values as needed.
# Using the max/min from the filtered data for dynamic bounds,
# or hardcode them if a fixed area is preferred.
lat_north = df_to_map['lat'].max() + 0.01  # Slightly north of max latitude
lat_south = df_to_map['lat'].min() - 0.01  # Slightly south of min latitude
long_east = df_to_map['lng'].max() + 0.01  # Slightly east of max longitude
long_west = df_to_map['lng'].min() - 0.01  # Slightly west of min longitude
# -----------------------------------------------------

# Create a map centered on the location
map_castello = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=13,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
).add_to(map_castello)

# Add a rectangle to the map using the defined bounds
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",
).add_to(map_castello)

# Add a circle marker for each photo
latitudes = df_to_map['lat']
longitudes = df_to_map['lng']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid",
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_castello)

map_castello

## Loop through all_place_id to save all reviews

In [ ]:
all_reviews = []

for pid, title, gps in zip(df_paginated_places['place_id'], df_paginated_places['title'], df_paginated_places['gps']):
    if not pid:
        continue

    print(f"Fetching reviews for: {title} ({pid})...")
    rev = serpapi_search('google_maps_reviews', {
        'place_id': pid,
        'hl': 'en',
        'sort_by': 'most_relevant'
    })

    reviews = rev.get('reviews', [])
    for r in reviews:
        all_reviews.append({
            'place_id': pid,
            'place_title': title,
            'gps': gps,  # ✅ keep coordinates
            'rating': r.get('rating'),
            'date': r.get('date'),
            'title': r.get('title'),
            'name': r.get('user', {}).get('name'),
            'review': r.get('snippet'),
            'images': r.get('images'),
            'likes': r.get('likes'),
            'local_guide': r.get('user', {}).get('local_guide'),
        })

df_all_reviews = pd.DataFrame(all_reviews)

Fetching reviews for: Conad (ChIJ_1sIlcaxfkcRoCg9GfYmL6o)...
Fetching reviews for: Coop (ChIJr6rjRNqxfkcR6MWYsM1pwTI)...
Fetching reviews for: Supermercato Coop (ChIJz9iqFrixfkcRK5g8t1KreiE)...
Fetching reviews for: Conad City - Supermarket (ChIJl14O4eqxfkcRTSK1pzkmshk)...
Fetching reviews for: Despar Teatro Italia (ChIJlYIMP8KxfkcRHcslK4IM_JU)...
Fetching reviews for: CONAD CITY (ChIJs4f3kcuxfkcR2X7NorkQ1X0)...
Fetching reviews for: Supermercato Despar Rialto (ChIJ6f-9T9qxfkcRnA0o_f8LfSQ)...
Fetching reviews for: Crai SanMarco Supermarket (ChIJp3E4MI-xfkcRMke-YEst-2g)...
Fetching reviews for: Supermercato DESPAR San Marco (ChIJuypz9F-xfkcRkm6SYPQiX_M)...
Fetching reviews for: CONAD CITY (ChIJ5SLI0rOvfkcRbDU8y08plWs)...
Fetching reviews for: Supermercato Coop (ChIJq_t62QixfkcRKQN7NSyKNvQ)...
Fetching reviews for: Coop (ChIJSSe3DMSxfkcR3hPIx87KSI4)...
Fetching reviews for: CONAD CITY (ChIJnfh-QKyxfkcR4nRlnMX3dBg)...
Fetching reviews for: Despar (ChIJ-aUb-u6xfkcRmEK5Dpevakw)...
Fetching 

In [ ]:
df_all_reviews.head(10)

,place_id,place_title,gps,rating,date,title,name,review,images,likes,local_guide
0,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",3.0,3 months ago,None,Becky W.,When we arrived in Venice this summer and need...,None,None,True
1,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",5.0,4 months ago,None,Didier Huyberechts,This well-maintained supermarket is a good siz...,None,None,True
2,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",4.0,a year ago,None,chef chua,"Beautiful chocolate place, nice gift but over ...",[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
3,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",5.0,2 years ago,None,D M,"Let’s face it, Venice is quite expensive to ea...",[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
4,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",5.0,5 months ago,None,Maria Arcuri,"Small store but good selection of fresh food, ...",[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
5,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",5.0,9 months ago,None,Drew Senior,Staying in an Airbnb so used this supermarket ...,None,None,True
6,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",5.0,a year ago,None,Christine Madison,A wonderful grocery store 5 minutes from Hotel...,[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
7,ChIJ_1sIlcaxfkcRoCg9GfYmL6o,Conad,"{'latitude': 45.439881299999996, 'longitude': ...",4.0,6 years ago,None,Dean Ku,Wonderful little grocery store. Nice staff. ...,[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
8,ChIJr6rjRNqxfkcR6MWYsM1pwTI,Coop,"{'latitude': 45.4364907, 'longitude': 12.3343808}",5.0,3 months ago,None,Jojee Khan,Stumbled across the Coop in San Marco (Riva de...,[https://lh3.googleusercontent.com/geougc-cs/A...,None,True
9,ChIJr6rjRNqxfkcR6MWYsM1pwTI,Coop,"{'latitude': 45.4364907, 'longitude': 12.3343808}",4.0,2 weeks ago,None,Lorenzo Quadrelli,The Coop near Ponte Rialto offers a pleasant e...,None,None,True


##Save the output to CSV

In [ ]:
# 💾 Save your DataFrame: change df_name and path
df_name = 'df_all_reviews'   # replace with the variable name you want to save
OUTPATH = "/content/drive/MyDrive/semester1_python_dataset/Google/Data_Scratch/supermarket_reviews.csv"
globals()[df_name].to_csv(OUTPATH, index=False)
OUTPATH


'/content/drive/MyDrive/semester1_python_dataset/Google/Data_Scratch/supermarket_reviews.csv'


**Change ideas:**  
- Change query to typologies (bookstores, galleries) or experiences (“instagram cafe”).  
- Add spatial sampling by moving `location` / using `ll` + `radius` (see docs).  
- Compare sentiment by time using `sort_by=newest` and grouping by month.
